In [ ]:
from pathlib import Path
import os
import shutil
# from lib.uo_cli_wrapper import UOCliWrapper

from urbanopt_des.uo_cli_wrapper import UOCliWrapper

# autoreload the dependencies here when they
# change (specifically the uo_cli_wrapper.py file)
%load_ext autoreload
%autoreload 2

In [ ]:
# Get the working directories
workdir = Path().resolve()
print(f"Working directory: {workdir}")

analysis_dir = workdir / "esbe_2026"
if not analysis_dir.exists():
    analysis_dir.mkdir()

template_data_dir = workdir / "data" / "templates"

print(f"template_data_dir: {template_data_dir}")
print(f"analysis_dir: {analysis_dir}")

# Get the number of usable cores for parallel processing by looking at the system (n-2)
num_usable_cores = os.cpu_count() - 2
print(f"Number of usable cores for parallel processing: {num_usable_cores}")

### Activity 3: REopt

In [ ]:
# This is the same as above, but in a new directory
activity_3_analysis_dir = analysis_dir / "activity_3"
if not activity_3_analysis_dir.exists():
    activity_3_analysis_dir.mkdir()

uo_coincident = UOCliWrapper(
    activity_3_analysis_dir, "coincident_pre", template_dir=template_data_dir
)

uo_coincident.create_example_coincident_project()

uo_coincident.create_scenarios("class_project_coincident.json")

# copy over to the coincident and diverse directories for activity 3
uo_coincident = uo_coincident.update_project_files("coincident")

# run sims faster
uo_coincident.set_number_parallel(num_usable_cores)

# copy over the weather files
uo_coincident.copy_over_weather()

# change weather file in mapper file
# uo_coincident.replace_weather_file_in_mapper(
#     "base_workflow.osw", "Lecco_IT_TMY", "ASHRAE 169-2013-4A"
# )

# Fix issues -- copy over the updated Baseline.rb mapper file
shutil.copy2(
    template_data_dir / "mappers" / "Baseline.rb",
    uo_coincident.project_path / "mappers" / "Baseline.rb",
)
# Copy over the base workflow from the template. This one includes all the EEMs that are needed in the class project since
# the class_project_workflow.json inherits from base_workflow.osw. This is a workaround to avoid having to patch the workflow file to add in the missing EEM steps.
shutil.copy2(
    template_data_dir / "mappers" / "base_workflow.osw",
    uo_coincident.project_path / "mappers" / "base_workflow.osw",
)

In [ ]:
# Run the baseline (again, new folder, so new data are needed)
uo_coincident.run("class_project_coincident.json", "baseline_scenario.csv")

# post process/visualize the baseline
uo_coincident.process_scenario("class_project_coincident.json", "baseline_scenario.csv")
uo_coincident.visualize_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)
uo_coincident.visualize_feature("class_project_coincident.json")

In [ ]:
# Create the scenario mapper file that is enabled with the REopt assumptions
uo_coincident.create_reopt_scenario(
    "class_project_coincident.json", "baseline_scenario.csv"
)

# Confirm in the REopt_baseline_scenario file that the REopt assumptions are now enabled (look for the REopt Assumptions section) in the file

In [ ]:
# Run the REopt baseline scenario
uo_coincident.run("class_project_coincident.json", "REopt_baseline_scenario.csv")

In [ ]:
# if the REopt errors with cert issues, then look at this help site,
#   But where do you run the bundle update command?
#   https://docs.urbanopt.net/developer_resources/known_issues.html

# Also, try to access the reopt site directly to make sure the API is correct
#  https://developer.nrel.gov/api/reopt/v1/?API_KEY=ganRGlzka5XeOnae21cepxb1vkIX57fCsGc6x2EZ

uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_baseline_scenario.csv",
    individual_features=False,
)
uo_coincident.process_reopt_scenario(
    "class_project_coincident.json",
    "REopt_baseline_scenario.csv",
    individual_features=True,
)